In [63]:
from unittest import result

import pandas as pd

In [64]:
ground_truth = pd.read_csv('data/ground_truth-new.csv')

In [65]:
ground_truth = ground_truth.to_dict(orient='records')

In [66]:
ground_truth[0]

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [67]:
from Module01_AgenticRAG.ingest import load_faq_data, built_index

documents =  load_faq_data()
documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)
len(documents_llm)

115

In [68]:
index = built_index(documents_llm)

In [69]:
results = index.search(ground_truth[0]['question'],
             num_results=5)

In [70]:
for r in results:
    print(f'{r['id']} == {ground_truth[0]['document']}: {ground_truth[0]['document'] == r['id']}')

74eb249bbf == 74eb249bbf: True
977bf7786c == 74eb249bbf: False
04919992b3 == 74eb249bbf: False
489dd1c9d9 == 74eb249bbf: False
cdc3b285e5 == 74eb249bbf: False


In [71]:
def compute_relevance_text(q):
    doc_id = q['document']
    results = index.search(q['question'],
                           num_results=5)
    relevance = []
    for d in results:
        relevance.append(int(d['id']==doc_id))
    return relevance

In [72]:
compute_relevance_text(ground_truth[0])

[1, 0, 0, 0, 0]

In [73]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [74]:
compute_relevance_total_text(ground_truth)

  0%|          | 0/395 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0,

In [75]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [76]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [77]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [78]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [79]:
relevance_total = compute_relevance_total(ground_truth, text_search)


  0%|          | 0/395 [00:00<?, ?it/s]

In [80]:
sample = relevance_total[:15]

In [81]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt+=1
    return cnt/len(relevance)

In [82]:
hit_rate(relevance_total)

0.6506329113924051

In [83]:
import numpy as np
def mean_reciprocal_rank(relevance):
    df = pd.DataFrame(relevance)
    first_hit_idx = df.idxmax(axis=1)
    has_hit = df.max(axis=1)>=1
    reciprocal_ranks = np.where(has_hit,1.0/(first_hit_idx+1),0.0)
    return reciprocal_ranks.mean()


In [84]:
mean_reciprocal_rank(relevance_total)

np.float64(0.5756118143459916)

In [85]:
def evaluate(ground_truth,text_search):
    relevance_total = compute_relevance_total(ground_truth, text_search)
    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mean_reciprocal_rank(relevance_total),
    }

In [86]:
evaluate(ground_truth,text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': np.float64(0.5756118143459916)}

In [87]:
def text_searchv2(query):
    boost_dict = {"question": 1, "section": 0.4}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [88]:
evaluate(ground_truth,text_searchv2)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6987341772151898, 'mrr': np.float64(0.6116455696202532)}

In [89]:
def search_boost(query,question_boost):
    boost_dict = {"question": question_boost, "section": 0.4}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [91]:
import numpy as np
for boost in np.linspace(0.1,5,30):
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.1: {'hit_rate': 0.6506329113924051, 'mrr': np.float64(0.528860759493671)}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.2689655172413793: {'hit_rate': 0.6860759493670886, 'mrr': np.float64(0.580168776371308)}


  0%|          | 0/395 [00:00<?, ?it/s]

...

boost=4.662068965517241: {'hit_rate': 0.6379746835443038, 'mrr': np.float64(0.5542194092827004)}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=4.8310344827586205: {'hit_rate': 0.6379746835443038, 'mrr': np.float64(0.5529535864978903)}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.6379746835443038, 'mrr': np.float64(0.55042194092827)}


In [92]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [93]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [94]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
18,2.0,4.0,0.1,0.754430,0.648945
33,5.0,10.0,0.1,0.749367,0.648692
34,5.0,10.0,0.2,0.754430,0.648650
3,1.0,2.0,0.1,0.749367,0.647637
35,5.0,10.0,0.5,0.749367,0.647637
19,2.0,4.0,0.2,0.749367,0.647637
4,1.0,2.0,0.2,0.746835,0.645696
20,2.0,4.0,0.5,0.746835,0.644852
7,1.0,4.0,0.2,0.744304,0.634388
6,1.0,4.0,0.1,0.744304,0.634304


In [95]:
def text_search(query):
    boost_dict = {
        "question": 2.0,
        "answer": 4.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )